# Rule-Based Chatbot — 02 Building the Bot

We TF-IDF-vectorize all example patterns. For a user message we compute cosine similarity to every pattern, take the best match's intent, and reply with its response. Below a similarity **threshold** the bot returns a safe fallback instead of guessing.

In [1]:
import utils, warnings; warnings.filterwarnings('ignore')
intents=utils.load_intents()
df=utils.intents_to_frame(intents)
bot=utils.RuleBasedChatbot(intents,threshold=0.2).fit(df['pattern'],df['tag'])
print('chatbot ready —',len(df),'patterns over',df.tag.nunique(),'intents')

chatbot ready — 77 patterns over 12 intents


## 1. A demo conversation

In [2]:
for msg in ['hello there','when do you open','can i pay with paypal','i forgot my password','where is your store','asdfqwer xyz']:
    resp,tag,score=bot.respond(msg)
    print(f'USER: {msg}')
    print(f' BOT: {resp}  [intent={tag}, sim={score}]\n')

USER: hello there
 BOT: Hello! How can I help you today?  [intent=greeting, sim=0.78]

USER: when do you open
 BOT: We are open Monday to Friday, 9am to 6pm.  [intent=hours, sim=0.607]

USER: can i pay with paypal
 BOT: We accept Visa, Mastercard, and PayPal.  [intent=payments, sim=1.0]

USER: i forgot my password
 BOT: You can reset your password from the login page via 'Forgot password'.  [intent=account, sim=1.0]

USER: where is your store
 BOT: Our main store is at 123 Market Street.  [intent=location, sim=1.0]

USER: asdfqwer xyz
 BOT: Sorry, I didn't quite understand that. Could you rephrase?  [intent=fallback, sim=0.0]



## 2. The confidence threshold

Gibberish or out-of-scope messages score low similarity and trigger the fallback — this is what stops the bot from confidently answering nonsense.

In [3]:
for msg in ['blah blah random','quantum physics lecture','thanks so much']:
    resp,tag,score=bot.respond(msg)
    print(f'{msg!r:35s} -> sim={score} intent={tag}')

'blah blah random'                  -> sim=0.0 intent=fallback
'quantum physics lecture'           -> sim=0.0 intent=fallback
'thanks so much'                    -> sim=0.618 intent=thanks


## 3. How matching works

The top similar patterns for a query show why an intent is chosen.

In [4]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
q='do you take credit cards'
sims=cosine_similarity(bot.vec.transform([q]),bot.P).ravel()
top=pd.DataFrame({'pattern':df['pattern'],'tag':df['tag'],'sim':sims}).sort_values('sim',ascending=False).head(5)
print('query:',q); print(top.to_string(index=False))

query: do you take credit cards
                    pattern      tag      sim
   do you take credit cards payments 1.000000
           what do you sell products 0.170103
         do you accept cash payments 0.165625
          when do you close    hours 0.164597
do you ship internationally shipping 0.159930
